<a href="https://colab.research.google.com/github/tlgus2882-ctrl/nlp_miniproject/blob/main/01_nsmc_polarity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01. NSMC 극성 분류 재현 (KoELECTRA)

네이버 영화 리뷰(NSMC)를 긍정/부정으로 나누는 **극성 분류**를 KoELECTRA로 재현한다.
여기서 학습한 모델은 이후 **2단계 학습**(극성 모델 → 감정 분류)의 출발점으로 쓴다.

- 참조 코드: [monologg/KoELECTRA](https://github.com/monologg/KoELECTRA) `finetune/` 의 NSMC 예제
- 데이터: [e9t/nsmc](https://github.com/e9t/nsmc) — 별점 1–4 → 부정(0), 9–10 → 긍정(1)
- 목표: KoELECTRA-base-v3 보고 정확도(약 90%대 초반) 재현
- 실행 환경: Colab T4 (런타임 → 런타임 유형 변경 → GPU)

In [1]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install transformers datasets accelerate

## 1. 설정

In [2]:
import os

# SMOKE=1 이면 작은 모델 + 소량 데이터로 코드가 끝까지 도는지만 확인한다.
SMOKE = os.environ.get("SMOKE") == "1"

MODEL_NAME = "monologg/koelectra-small-v3-discriminator" if SMOKE else "monologg/koelectra-base-v3-discriminator"
MAX_LEN = 64        # NSMC 리뷰는 대부분 짧아 64 토큰이면 충분하다
EPOCHS = 2
LR = 5e-5
BATCH_SIZE = 64
SEED = 42

if IN_COLAB and not SMOKE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/miniproject/nsmc-polarity"  # 런타임이 끊겨도 2단계 학습에서 불러올 수 있게 Drive에 저장
else:
    OUTPUT_DIR = "outputs/nsmc-polarity"

Mounted at /content/drive


## 2. 데이터 불러오기

In [3]:
import pandas as pd

NSMC_URL = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_{}.txt"

def load_nsmc(split):
    df = pd.read_csv(NSMC_URL.format(split), sep="\t", quoting=3)  # quoting=3: 리뷰 속 따옴표를 그대로 둔다
    df = df.dropna(subset=["document"])
    return df[["document", "label"]].reset_index(drop=True)

train_df = load_nsmc("train")
test_df = load_nsmc("test")

if SMOKE:
    train_df = train_df.sample(200, random_state=SEED).reset_index(drop=True)
    test_df = test_df.sample(100, random_state=SEED).reset_index(drop=True)

print(f"train {len(train_df):,} / test {len(test_df):,}")
print(train_df["label"].value_counts(normalize=True).rename({0: "부정", 1: "긍정"}))
train_df.head()

train 149,995 / test 49,997
label
부정    0.50115
긍정    0.49885
Name: proportion, dtype: float64


,document,label
0,아 더빙.. 진짜 짜증나네요 목소리,0
1,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,너무재밓었다그래서보는것을추천한다,0
3,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


## 3. 토크나이즈

In [4]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["document"], truncation=True, max_length=MAX_LEN)

train_ds = Dataset.from_pandas(train_df).map(tokenize, batched=True).rename_column("label", "labels")
test_ds = Dataset.from_pandas(test_df).map(tokenize, batched=True).rename_column("label", "labels")

print(tokenizer.tokenize(train_df["document"][0]))

config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/61.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/263k [00:00<?, ?B/s]

Map:   0%|          | 0/149995 [00:00<?, ? examples/s]

Map:   0%|          | 0/49997 [00:00<?, ? examples/s]

['아', '더빙', '.', '.', '진짜', '짜증', '##나', '##네', '##요', '목소리']


## 4. 모델 구조

`AutoModelForSequenceClassification`은 ELECTRA에서 `ElectraForSequenceClassification`이 된다.

| 부분 | 코드상 위치 | 역할 |
|---|---|---|
| ELECTRA 판별기 (인코더) | `model.electra` | 사전학습에서 "각 토큰이 원본인지 생성기가 바꾼 것인지" 맞히도록 학습된 Transformer 인코더. 문장을 토큰별 벡터로 바꾼다 |
| 분류 헤드 | `model.classifier` | 첫 토큰(`[CLS]`) 벡터 → Dense → GELU → Dropout → `out_proj`(hidden → `num_labels`) |
| 손실 | `forward` 내부 | `num_labels=2`, 정수 라벨이면 softmax + cross-entropy (**단일 라벨**) |

감정 분류로 확장할 때 바꾸는 곳은 분류 헤드의 출력 크기(2 → 8)와 손실(softmax CE → sigmoid BCE, **다중 라벨**)이다.
2단계 학습에서는 여기서 학습한 `model.electra`(인코더)를 그대로 가져가고 헤드만 새로 만든다.

In [5]:
import torch
from transformers import AutoModelForSequenceClassification, set_seed

set_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label={0: "부정", 1: "긍정"}, label2id={"부정": 0, "긍정": 1}
)

print(model.classifier)
n_params = sum(p.numel() for p in model.parameters())
n_head = sum(p.numel() for p in model.classifier.parameters())
print(f"전체 파라미터 {n_params:,} 중 분류 헤드 {n_head:,} ({n_head / n_params:.2%})")

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  452MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  452MB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] ElectraForSequenceClassification LOAD REPORT from: monologg/koelectra-base-v3-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.out_proj.weight                        | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.dense.weight                           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your do

ElectraClassificationHead(
  (dense): Linear(in_features=768, out_features=768, bias=True)
  (activation): GELUActivation()
  (dropout): Dropout(p=0.1, inplace=False)
  (out_proj): Linear(in_features=768, out_features=2, bias=True)
)
전체 파라미터 112,922,882 중 분류 헤드 592,130 (0.52%)


## 5. 학습

In [6]:
import numpy as np
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    return {"accuracy": (np.argmax(logits, axis=-1) == labels).mean()}

import math
total_steps = math.ceil(len(train_ds) / BATCH_SIZE) * EPOCHS

args = TrainingArguments(
    output_dir="checkpoints/nsmc-polarity",
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    warmup_steps=int(total_steps * 0.1),  # 전체 스텝의 10% 동안 학습률을 서서히 올린다
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_steps=200,
    save_strategy="no",
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    data_collator=DataCollatorWithPadding(tokenizer),  # 배치마다 가장 긴 문장 길이로만 패딩
    compute_metrics=compute_metrics,
)

trainer.train()

Step,Training Loss
200,0.543759
400,0.329739
600,0.306390
800,0.288327
1000,0.279395
1200,0.266201
1400,0.257129
1600,0.250704
1800,0.246076
2000,0.253969


TrainOutput(global_step=4688, training_loss=0.2331886055526473, metrics={'train_runtime': 871.1772, 'train_samples_per_second': 344.35, 'train_steps_per_second': 5.381, 'total_flos': 9824435251614720.0, 'train_loss': 0.2331886055526473, 'epoch': 2.0})

## 6. 평가

In [7]:
result = trainer.evaluate(test_ds)
print(f"NSMC test 정확도: {result['eval_accuracy']:.4f}")

Training Loss,Validation Loss,Step,Accuracy
0.167268,0.238480,4688,0.910855


NSMC test 정확도: 0.9109


## 7. 저장 후 다시 불러와 예측

In [8]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

from transformers import pipeline

classifier = pipeline("text-classification", model=OUTPUT_DIR, device=0 if torch.cuda.is_available() else -1)

reviews = [
    "배우들 연기가 미쳤다. 두 번 봐도 또 보고 싶음",
    "시간 아깝다 진짜 노잼",
    "결말이 너무 어이없어서 화가 난다",
    "음악이 좋아서 끝까지 봤다",
]
for review, pred in zip(reviews, classifier(reviews)):
    print(f"{pred['label']} ({pred['score']:.2f})  {review}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

긍정 (1.00)  배우들 연기가 미쳤다. 두 번 봐도 또 보고 싶음
부정 (1.00)  시간 아깝다 진짜 노잼
부정 (0.99)  결말이 너무 어이없어서 화가 난다
부정 (0.96)  음악이 좋아서 끝까지 봤다
